# Model — ตอบคำถามธุรกิจกลุ่ม B

โหลด **dataset #2 (พร้อมเทรนโมเดล)** จาก Hugging Face แล้วทำโมเดลตัวอย่าง 3 แบบ
ที่ครอบคลุมตระกูลหลัก:

| # | คำถาม | โมเดล |
|---|-------|-------|
| B2 | ข้อความในรีวิวตรงกับดาวที่ให้ไหม | **Classification** (TF-IDF + Logistic Regression) |
| B3 | สินค้าแบ่งได้กี่กลุ่มตาม performance | **Clustering** (K-Means) |
| B1 | สินค้าใหม่จะได้คะแนนเท่าไหร่ | **Regression** (Ridge เทียบ Gradient Boosting) |

ทั้งหมดรันบน CPU ได้ ไม่ต้องใช้ GPU

**หลักที่ยึดทุกข้อ**
- ใช้ split ที่เตรียมมาแล้ว (`train`/`val`/`test`) — **ห้ามสุ่มแบ่งใหม่** เพราะจะกลายเป็น
  การทำนายอดีตจากอนาคต (data leakage)
- เทียบกับ **baseline** เสมอ ถ้าชนะ baseline ไม่ได้ต้องรายงานตามตรง
- คลาสไม่สมดุลหนัก → ใช้ `macro-F1` ไม่ใช่ `accuracy` และตั้ง `class_weight='balanced'`

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from elt.common import load_config, resolve_dataset  # noqa: E402

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

config = load_config()
SRC = resolve_dataset("preprocessing", config)

con = duckdb.connect()
con.execute(f"CREATE VIEW reviews AS SELECT * FROM '{SRC}/reviews/**/*.parquet'")
con.execute(f"CREATE VIEW product_features AS SELECT * FROM '{SRC}/product_features.parquet'")

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

con.sql("""
    SELECT split, count(*) AS n_reviews,
           min(review_date) AS first_date, max(review_date) AS last_date,
           round(avg(rating), 2) AS avg_rating
    FROM reviews GROUP BY 1 ORDER BY first_date
""").df()

## B2. ข้อความในรีวิวตรงกับดาวที่ให้ไหม? — Classification

**ทำไมต้องใช้โมเดล** — ต้องเข้าใจความหมายของข้อความ ไม่มีคอลัมน์ไหนบอกได้ว่า
รีวิวนี้ "น้ำเสียงบวกหรือลบ"

**ประโยชน์จริง** ไม่ได้อยู่ที่ค่า F1 แต่อยู่ที่การเอาโมเดลไปหา**รีวิวที่โมเดลมั่นใจว่าลบ
แต่ลูกค้าให้ 5 ดาว** (และกลับกัน) — กลุ่มนี้คือคนที่กดดาวผิด ประชด หรือให้ดาวตามของแถม
ซึ่งทำให้คะแนนสินค้าเพี้ยน

> สุ่มตัวอย่างจาก train เพื่อให้รันจบใน 1–2 นาทีบนโน้ตบุ๊ก
> ถ้าจะใช้จริงให้เอาออก หรือเปลี่ยนไป fine-tune DistilBERT

In [ ]:
TRAIN_SAMPLE = 300_000        # ตั้ง None ถ้าอยากใช้ทั้งหมด (2.58M แถว)
TEST_SAMPLE = 100_000

def load_split(split, limit=None, cols="text_full, sentiment, rating, review_id, category"):
    q = f"SELECT {cols} FROM reviews WHERE split = '{split}'"
    if limit:
        q += f" USING SAMPLE {limit} ROWS (reservoir, {RANDOM_STATE})"
    return con.sql(q).df()

train = load_split("train", TRAIN_SAMPLE)
test  = load_split("test",  TEST_SAMPLE)
print(f"train {len(train):,} แถว | test {len(test):,} แถว")

dist = pd.DataFrame({
    "train": train.sentiment.value_counts(normalize=True),
    "test":  test.sentiment.value_counts(normalize=True),
}).round(3)
print("\nสัดส่วนคลาส — สังเกตว่า test มีรีวิวลบมากกว่า train (distribution shift ของจริง):")
dist

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
)
clf.fit(train.text_full, train.sentiment)

pred = clf.predict(test.text_full)
print(classification_report(test.sentiment, pred, digits=3))

# baseline: ทายคลาสที่พบบ่อยที่สุดเสมอ — ถ้าโมเดลไม่ชนะอันนี้ก็ไม่มีประโยชน์
majority = train.sentiment.mode()[0]
base_f1 = f1_score(test.sentiment, [majority] * len(test), average="macro", zero_division=0)
model_f1 = f1_score(test.sentiment, pred, average="macro")
print(f"macro-F1  baseline (ทาย '{majority}' เสมอ): {base_f1:.3f}")
print(f"macro-F1  โมเดล:                        {model_f1:.3f}")
print(f"→ ดีขึ้น {model_f1 - base_f1:+.3f}")

In [ ]:
labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(test.sentiment, pred, labels=labels, normalize="true")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), labels); ax.set_yticks(range(3), labels)
ax.set_xlabel("ทำนาย"); ax.set_ylabel("จริง")
ax.set_title("Confusion matrix (normalize ตามแถว)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm[i, j] > 0.5 else "black")
ax.grid(False); plt.colorbar(im); plt.tight_layout(); plt.show()

print("neutral (3 ดาว) มักถูกทายผิดที่สุด — เป็นเรื่องปกติ")
print("เพราะข้อความ 3 ดาวมีทั้งคำชมและคำติปนกัน ขอบเขตจึงคลุมเครือโดยธรรมชาติ")

In [ ]:
# ── ประโยชน์ทางธุรกิจที่แท้จริงของข้อนี้ ──
# หารีวิวที่โมเดลมั่นใจมากว่า "ลบ" แต่ลูกค้าให้ 5 ดาว
proba = clf.predict_proba(test.text_full)
neg_idx = list(clf.classes_).index("negative")

test = test.assign(p_negative=proba[:, neg_idx])
mismatch = test[(test.rating == 5) & (test.p_negative > 0.9)].nlargest(5, "p_negative")

print(f"รีวิว 5 ดาวที่โมเดลมั่นใจ >90% ว่าเป็นข้อความเชิงลบ: "
      f"{((test.rating == 5) & (test.p_negative > 0.9)).sum():,} จาก {len(test):,} แถว\n")
for _, r in mismatch.iterrows():
    print(f"[{r.p_negative:.0%} ลบ | ให้ {int(r.rating)} ดาว] {r.text_full[:150]}...\n")

## B3. สินค้าแบ่งได้เป็นกี่กลุ่มตามลักษณะ performance? — Clustering

**ทำไมต้องใช้โมเดล** — เราไม่รู้ล่วงหน้าว่ามีกี่กลุ่มและแบ่งด้วยเกณฑ์อะไร
ถ้าตั้งกฎเอง (เช่น "คะแนน > 4 = ดี") ก็แค่ยัดความเชื่อของเราลงไป

**สิ่งที่ต้องทำก่อนเสมอ** — standardize เพราะ K-Means ไวต่อสเกลมาก
ถ้าไม่ทำ `n_reviews` (หลักพัน) จะกลบ `avg_rating` (หลักหน่วย) จนคลัสเตอร์ไร้ความหมาย

In [ ]:
# ตัดสินค้าที่รีวิวน้อยออก เพราะสถิติยังไม่นิ่ง (คะแนนเฉลี่ยจาก 2 รีวิวเชื่อไม่ได้)
MIN_REVIEWS = 20

prod = con.sql(f"""
    SELECT product_key, category, product_title, store, price_band,
           n_reviews, avg_rating, rating_std, negative_share,
           polarized_share, verified_share, review_span_days
    FROM product_features
    WHERE n_reviews >= {MIN_REVIEWS} AND rating_std IS NOT NULL
""").df()
print(f"สินค้าที่เข้าเกณฑ์: {len(prod):,} จากทั้งหมด "
      f"{con.execute('SELECT count(*) FROM product_features').fetchone()[0]:,}")

FEATURES = ["avg_rating", "rating_std", "negative_share",
            "polarized_share", "verified_share", "review_span_days"]
X = prod[FEATURES].copy()
X["n_reviews_log"] = np.log1p(prod.n_reviews)   # เบ้ขวาหนัก ต้อง log ก่อน
X.describe().round(3)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)

# เลือก k ด้วย elbow + silhouette (silhouette คำนวณบน subsample เพื่อความเร็ว)
sub = np.random.choice(len(Xs), min(10_000, len(Xs)), replace=False)
scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(Xs)
    scores.append((k, km.inertia_, silhouette_score(Xs[sub], km.labels_[sub])))

sc = pd.DataFrame(scores, columns=["k", "inertia", "silhouette"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(sc.k, sc.inertia, marker="o"); ax1.set_title("Elbow"); ax1.set_xlabel("k")
ax2.plot(sc.k, sc.silhouette, marker="o", color="darkorange")
ax2.set_title("Silhouette (สูงกว่า = ดีกว่า)"); ax2.set_xlabel("k")
plt.tight_layout(); plt.show()
sc.round(3)

In [ ]:
K = 4        # ปรับตามกราฟข้างบน
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10).fit(Xs)
prod["cluster"] = km.labels_

profile = prod.groupby("cluster").agg(
    n_products=("product_key", "size"),
    avg_rating=("avg_rating", "mean"),
    rating_std=("rating_std", "mean"),
    negative_share=("negative_share", "mean"),
    polarized_share=("polarized_share", "mean"),
    verified_share=("verified_share", "mean"),
    median_reviews=("n_reviews", "median"),
).round(3)
display(profile)

# ตั้งชื่อกลุ่มจากลักษณะเด่น แทนที่จะเรียก "cluster 0/1/2"
print("อ่านตารางแล้วตั้งชื่อกลุ่มเอง เช่น:")
print("  คะแนนสูง + sd ต่ำ           -> 'ดาวเด่นมั่นคง'")
print("  คะแนนกลาง + polarized สูง   -> 'แตกขั้ว (ความคาดหวังไม่ตรง)'")
print("  negative_share สูง          -> 'ต้องเข้าไปแก้ด่วน'")
print("  n_reviews น้อย + span สั้น   -> 'ยังใหม่ ข้อมูลไม่พอตัดสิน'")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
sc_ = ax.scatter(prod.avg_rating, prod.rating_std, c=prod.cluster,
                 cmap="tab10", s=6, alpha=0.35)
ax.set_xlabel("คะแนนเฉลี่ย"); ax.set_ylabel("ส่วนเบี่ยงเบนมาตรฐานของคะแนน")
ax.set_title(f"สินค้า {len(prod):,} ชิ้น แบ่งเป็น {K} กลุ่ม")
plt.colorbar(sc_, label="cluster"); plt.tight_layout(); plt.show()

print("ตัวอย่างสินค้าในแต่ละกลุ่ม:")
for c in sorted(prod.cluster.unique()):
    r = prod[prod.cluster == c].nlargest(1, "n_reviews").iloc[0]
    print(f"  กลุ่ม {c} | {r.avg_rating:.2f}⭐ sd={r.rating_std:.2f} "
          f"{r.n_reviews:>4} รีวิว | {str(r.product_title)[:45]}")

## B1. สินค้าใหม่จะได้คะแนนเท่าไหร่? — Regression

**ทำไมต้องใช้โมเดล** — สินค้าใหม่ยัง**ไม่มีรีวิวเลย** จะ `GROUP BY` หาค่าเฉลี่ยของมันไม่ได้
ต้องเรียนรู้ความสัมพันธ์จากสินค้าที่มีอยู่แล้ว

⚠️ **สองกับดักที่ต้องระวัง**
1. **แบ่งตามเวลา** ด้วย `first_review_date` ไม่ใช่สุ่ม — ไม่งั้นคือทำนายอดีตจากอนาคต
2. **ห้ามใช้** `avg_rating` ของตัวเองหรือ `listed_avg_rating` เป็น feature — นั่นคือคำตอบ

In [ ]:
# ใช้เฉพาะสินค้าที่รู้ราคา (feature หลัก) และมีรีวิวพอให้ target นิ่ง
reg = con.sql("""
    SELECT category, store, price, price_band, n_reviews,
           first_review_date, avg_rating
    FROM product_features
    WHERE price IS NOT NULL AND n_reviews >= 20
""").df()

# แบ่งตามเวลา: สินค้าที่เปิดตัวก่อน 2019 = train, ตั้งแต่ 2019 = test
cutoff = pd.Timestamp("2019-01-01")
reg["first_review_date"] = pd.to_datetime(reg.first_review_date)
tr, te = reg[reg.first_review_date < cutoff], reg[reg.first_review_date >= cutoff]
print(f"train {len(tr):,} สินค้า (เปิดตัวก่อน 2019) | test {len(te):,} สินค้า (2019+)")

# encode แบรนด์ด้วยความถี่ ไม่ใช่ target encoding เพราะ target encoding รั่วง่าย
freq = tr.store.value_counts()

def featurize(df):
    out = pd.DataFrame(index=df.index)
    out["price_log"] = np.log1p(df.price)
    out["store_freq"] = df.store.map(freq).fillna(0)
    for c in sorted(reg.category.unique()):
        out[f"cat_{c}"] = (df.category == c).astype(int)
    for b in sorted(reg.price_band.unique()):
        out[f"band_{b}"] = (df.price_band == b).astype(int)
    return out

Xtr, ytr = featurize(tr), tr.avg_rating
Xte, yte = featurize(te), te.avg_rating
print(f"feature: {Xtr.shape[1]} คอลัมน์")

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

results = []
# baseline ที่ต้องเอาชนะให้ได้: ทายค่าเฉลี่ยของ train เสมอ
results.append(("Baseline (ทายค่าเฉลี่ย)",
                mean_absolute_error(yte, np.full(len(yte), ytr.mean())), 0.0))

for name, model in [("Ridge", Ridge(alpha=1.0)),
                    ("GradientBoosting", HistGradientBoostingRegressor(
                        random_state=RANDOM_STATE, max_iter=200))]:
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    results.append((name, mean_absolute_error(yte, p), r2_score(yte, p)))

res = pd.DataFrame(results, columns=["model", "MAE", "R2"]).round(4)
display(res)

best = res.iloc[1:].nsmallest(1, "MAE").iloc[0]
gain = res.MAE.iloc[0] - best.MAE
print(f"โมเดลดีที่สุด: {best.model}  MAE {best.MAE:.4f}")
print(f"ดีกว่า baseline {gain:.4f} ดาว ({gain / res.MAE.iloc[0]:.1%})")
if gain < 0.01:
    print("\n→ แทบไม่ชนะ baseline: ราคา+แบรนด์+หมวด อธิบายคะแนนได้น้อยมาก")
    print("  ต้องรายงานตามตรง และถ้าจะทำต่อควรเพิ่ม feature จากข้อความรายละเอียดสินค้า")

In [ ]:
best_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE, max_iter=200).fit(Xtr, ytr)
pred_te = best_model.predict(Xte)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(pred_te, yte, s=5, alpha=0.2)
lim = [min(pred_te.min(), yte.min()), max(pred_te.max(), yte.max())]
ax1.plot(lim, lim, "r--", lw=1)
ax1.set_xlabel("ทำนาย"); ax1.set_ylabel("จริง"); ax1.set_title("ทำนาย vs จริง")

ax2.hist(yte - pred_te, bins=50, color="steelblue")
ax2.axvline(0, color="red", ls="--", lw=1)
ax2.set_xlabel("ค่าจริง - ค่าทำนาย"); ax2.set_title("การกระจายของ residual")
plt.tight_layout(); plt.show()

print("ถ้าจุดกระจุกเป็นแนวนอน (ทำนายค่าใกล้เคียงกันหมด) แปลว่าโมเดลแทบไม่ได้เรียนรู้อะไร")
print("ซึ่งเป็นผลลัพธ์ที่ถูกต้องและต้องรายงาน ไม่ใช่ความผิดพลาด")

## สรุปและข้อควรระวัง

| โมเดล | ผลที่ควรได้ | ใช้ต่อยังไง |
|-------|-------------|------------|
| B2 Classification | macro-F1 ชนะ baseline ชัดเจน | หารีวิวที่ดาวไม่ตรงข้อความ → ทำความสะอาดคะแนนสินค้า |
| B3 Clustering | 3–5 กลุ่มที่ตีความได้ | ตั้งกลยุทธ์รายกลุ่ม |
| B1 Regression | ชนะ baseline **เล็กน้อย** | ใช้เป็นสัญญาณคร่าว ๆ ไม่ใช่ตัวตัดสิน |

**ข้อควรระวังที่ต้องเขียนในรายงานเสมอ**

1. **B1 อ้างได้เฉพาะสินค้าที่รู้ราคา** (~21% ของรีวิว) ซึ่งเป็นกลุ่มที่คะแนนสูงกว่าค่าเฉลี่ยอยู่แล้ว
2. **B2 มี distribution shift** — สัดส่วนรีวิวลบใน test สูงกว่า train
   คะแนนบน test ที่ต่ำกว่า validation เป็นเรื่องปกติ ไม่ใช่บั๊ก
3. **B3 คลัสเตอร์ไม่ใช่ความจริงสัมบูรณ์** — เปลี่ยน k หรือ feature แล้วกลุ่มเปลี่ยน
   ต้องตรวจว่ากลุ่มที่ได้ "ตีความเป็นภาษาคนได้" ก่อนเอาไปใช้
4. **ไม่มีข้อมูลยอดขาย** — ทุกข้อสรุปพูดได้แค่เรื่องคะแนนและรีวิว ไม่ใช่รายได้

ทำต่อได้อีก: **B4** topic modeling บนรีวิวเชิงลบ (BERTopic) และ **B5** anomaly detection
หารีวิวน่าสงสัย — ดูวิธีตั้งโจทย์ใน [`../docs/business_questions.md`](../docs/business_questions.md)

In [ ]:
con.close()
print("model notebook เสร็จเรียบร้อย")